# 06 — BA-Level LMP Choropleth Map

**Purpose:** Interactive Folium choropleth of mean zonal LMPs for all completed
E4ST scenarios.  One `FeatureGroup` per scenario; a `LayerControl` widget lets
the user toggle between them at runtime.

Architecture note: this is a copper-plate zonal model — there is no nodal
congestion and therefore no congestion-rent layer.  Every bus inside a BA zone
shares a single price determined by the zone-level power-balance shadow price.

Color scale: ColorBrewer YlOrRd (9-class, perceptually uniform sequential)
mapped linearly from the 2nd to 98th percentile of all scenario LMPs so that
layers are visually comparable.

**Inputs:**
- `data/processed/e4st_results/{scenario}/lmp.parquet`
- `data/processed/ba_territories.geojson`
- `data/processed/network_metadata.json`

**Outputs:**
- `data/processed/figures/lmp_map.html`

In [ ]:
import sys, json
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import pandas as pd
import geopandas as gpd
import folium
import branca.colormap as cm_b
import numpy as np

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
RESULTS_DIR = PROJECT_ROOT / 'data' / 'processed' / 'e4st_results'
BA_GEO_PATH = PROJECT_ROOT / 'data' / 'processed' / 'ba_territories.geojson'
META_PATH   = PROJECT_ROOT / 'data' / 'processed' / 'network_metadata.json'
FIGURES_DIR = PROJECT_ROOT / 'data' / 'processed' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

with open(META_PATH) as f:
    meta = json.load(f)

SCENARIOS = [
    s['name'] for s in meta['scenarios_completed'] if s['status'] == 'OPTIMAL'
]
SCENARIO_LABELS = {
    'baseline':       'Baseline',
    'carbon_tax_50':  'Carbon Tax $50/tCO\u2082',
    'ces_achievable': 'CES 50% (max feasible)',
}
print('Scenarios to map:', SCENARIOS)

In [ ]:
# ── Load LMP parquets + BA territory geometries ────────────────────────────────
lmp_frames = {sc: pd.read_parquet(RESULTS_DIR / sc / 'lmp.parquet') for sc in SCENARIOS}

ba_gdf = gpd.read_file(BA_GEO_PATH)[['ba_code', 'ba_name', 'geometry']]

# Inner-join each scenario's LMPs onto the BA polygons (67 modelled zones)
merged = {}
for sc, df in lmp_frames.items():
    g = ba_gdf.merge(df[['ba', 'lmp_mwh']], left_on='ba_code', right_on='ba', how='inner')
    merged[sc] = g
    print(f'{sc}: {len(g)} zones matched, '
          f'LMP ${g["lmp_mwh"].min():.2f}\u2013${g["lmp_mwh"].max():.2f}/MWh')

# ── Shared color scale: 2nd–98th pct across ALL scenarios ─────────────────────
all_lmps = pd.concat([df['lmp_mwh'] for df in lmp_frames.values()])
vmin = float(all_lmps.quantile(0.02))
vmax = float(all_lmps.quantile(0.98))
print(f'\nShared scale: ${vmin:.2f}\u2013${vmax:.2f}/MWh')

In [ ]:
# ── ColorBrewer YlOrRd colormap via branca ────────────────────────────────────
# branca.colormap.linear.YlOrRd_09 is the 9-class ColorBrewer sequential ramp
colormap = cm_b.linear.YlOrRd_09.scale(vmin, vmax)
colormap.caption = 'Mean LMP ($/MWh)'

# Build per-scenario BA→hex lookup so style_function closures capture a dict
# (avoids the Python late-binding closure trap)
sc_colors = {}
for sc, gdf in merged.items():
    sc_colors[sc] = {
        row['ba_code']: colormap(row['lmp_mwh'])
        for _, row in gdf.iterrows()
    }

print('Sample colors (baseline):')
for ba, hex_c in list(sc_colors['baseline'].items())[:4]:
    print(f'  {ba}: {hex_c}')

In [ ]:
# ── Build Folium map ───────────────────────────────────────────────────────────
m = folium.Map(location=[39.5, -98.35], zoom_start=4, tiles='CartoDB positron')

for sc in SCENARIOS:
    gdf      = merged[sc]
    colors   = sc_colors[sc]
    label    = SCENARIO_LABELS.get(sc, sc)
    fg       = folium.FeatureGroup(name=label, show=(sc == 'baseline'))

    def make_style(color_lookup):
        """Return a style_function that reads pre-computed hex colours."""
        def style_fn(feature):
            ba  = feature['properties'].get('ba_code', '')
            hex_c = color_lookup.get(ba, '#cccccc')
            return {
                'fillColor':   hex_c,
                'color':       '#555555',
                'weight':       0.6,
                'fillOpacity':  0.78,
            }
        return style_fn

    for _, row in gdf.iterrows():
        lmp_val = row['lmp_mwh']
        folium.GeoJson(
            data={
                'type': 'Feature',
                'geometry': row.geometry.__geo_interface__,
                'properties': {'ba_code': row['ba_code']},
            },
            style_function=make_style(colors),
            tooltip=folium.Tooltip(
                f'<b>{row["ba_code"]}</b> \u2014 {row["ba_name"]}<br>'
                f'LMP: <b>${lmp_val:.2f}/MWh</b>',
                sticky=True,
            ),
        ).add_to(fg)

    fg.add_to(m)

# ── Add colorbar legend ────────────────────────────────────────────────────────
colormap.add_to(m)

# ── Layer toggle ───────────────────────────────────────────────────────────────
folium.LayerControl(collapsed=False).add_to(m)

out = FIGURES_DIR / 'lmp_map.html'
m.save(str(out))
print(f'Saved \u2192 {out}')
m

In [ ]:
# ── LMP summary table ─────────────────────────────────────────────────────────
rows = []
for sc in SCENARIOS:
    df = lmp_frames[sc]
    rows.append({
        'Scenario':           SCENARIO_LABELS.get(sc, sc),
        'Mean ($/MWh)':       df['lmp_mwh'].mean(),
        'Min ($/MWh)':        df['lmp_mwh'].min(),
        'Max ($/MWh)':        df['lmp_mwh'].max(),
        'Std Dev ($/MWh)':    df['lmp_mwh'].std(),
        'IQR ($/MWh)':        df['lmp_mwh'].quantile(0.75) - df['lmp_mwh'].quantile(0.25),
    })
pd.DataFrame(rows).set_index('Scenario').style.format('{:.2f}')